## Import Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, upper, lpad, regexp_replace, lower, substring, when, concat

## Reading from bronze layer

In [0]:
df = spark.table("olist.bronze.customers")
df.display()

## Overview about the table

In [0]:
print("=== Schema ===")
df.printSchema()

print("=== Row Count ===")
print(f"Total rows: {df.count()}")

print("=== Null Counts per Column ===")
df.select([
    F.count(F.when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()

## Transformations

### 1. TRIM whitespace from all string columns

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### 2. Normalize improperly represented nulls in string columns

In [0]:
NULL_STRINGS = ["", "null", "none", "n/a", "na", "unknown", "-", " "]

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(
            field.name,
            when(lower(trim(col(field.name))).isin(NULL_STRINGS), None)
            .otherwise(col(field.name))
        )

### 3. Fix customer_zip_code_prefix - pad with leading zeros to 5 digits

In [0]:
df = df.withColumn(
  "customer_zip_code_prefix", 
  lpad(col("customer_zip_code_prefix"), 5, "0")
)

### 4. Standardize customer_state - UPPERCASE 

In [0]:
df = df.withColumn(
    "customer_state",
    upper(trim(col("customer_state")))
)

### 5. Standardize customer_city - Title Case & normalize spaces

In [0]:
# Fixed
df = df.withColumn(
    "customer_city",
    trim(regexp_replace(col("customer_city"), r"\s+", " "))
)
df = df.withColumn(
    "customer_city",
    concat(
      upper(substring(col("customer_city"), 1, 1)), 
      lower(substring(col("customer_city"), 2, 9999))
    )
)

### 6. Handle nulls - filter out rows missing critical keys

In [0]:
df = df.filter(
    col("customer_id").isNotNull() &
    col("customer_unique_id").isNotNull() 
    )


### 7. Remove duplicate customer_id rows

In [0]:
df = df.dropDuplicates(["customer_id"])


## Preview quality check

In [0]:
print(f"Total rows after cleaning: {df.count()}")
print(f"Unique customers: {df.select('customer_id').distinct().count()}")
print(f"Unique states: {df.select('customer_state').distinct().count()}")
print(f"Null customer_city: {df.filter(col('customer_city').isNull()).count()}")
print(f"Null customer_state: {df.filter(col('customer_state').isNull()).count()}")

df.display()

## Write to silver layer

In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("olist.silver.customers")